In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix


: 

In [ ]:
#Load Data
base = Path.cwd()
if not (base / "data").exists():
    base = base.parent

X_train = pd.read_csv(base / "data/processed/train/X_train.csv")
X_val   = pd.read_csv(base / "data/processed/val/X_val.csv")
X_test  = pd.read_csv(base / "data/processed/test/X_test.csv")

y_train = pd.read_csv(base / "data/processed/train/y_train.csv").values.ravel()
y_val   = pd.read_csv(base / "data/processed/val/y_val.csv").values.ravel()
y_test  = pd.read_csv(base / "data/processed/test/y_test.csv").values.ravel()

print("Data loaded.")

In [ ]:
#Subsample for PSO speed
SAMPLE_SIZE = 50000
np.random.seed(42)
idx = np.random.choice(len(X_train), SAMPLE_SIZE, replace=False)
X_sample = X_train.iloc[idx]
y_sample = y_train[idx]
print(f"Subsampled {SAMPLE_SIZE} rows for PSO evaluation")

#Hyperparameter Search Space
# PSO tunes these 4 RF hyperparameters:
# Dimension 0: n_estimators     [50, 300]
# Dimension 1: max_depth        [5, 30]
# Dimension 2: min_samples_split[2, 20]
# Dimension 3: min_samples_leaf [1, 10]

BOUNDS_LOW  = np.array([50,  5, 2, 1], dtype=float)
BOUNDS_HIGH = np.array([300, 30, 20, 10], dtype=float)
N_DIMS = len(BOUNDS_LOW)

def decode_particle(pos):
    """Convert continuous position to valid RF hyperparameters."""
    pos = np.clip(pos, BOUNDS_LOW, BOUNDS_HIGH)
    return {
        "n_estimators":      int(round(pos[0])),
        "max_depth":         int(round(pos[1])),
        "min_samples_split": int(round(pos[2])),
        "min_samples_leaf":  int(round(pos[3]))
    }


In [ ]:
#PSO Configuration
N_PARTICLES = 15
N_ITERATIONS = 20
W  = 0.7    # inertia weight — controls how much particle keeps current direction
C1 = 1.5   # cognitive coefficient — pull toward personal best
C2 = 1.5   # social coefficient   — pull toward global best

np.random.seed(42)

In [ ]:
#PSO Fitness Function
def fitness(pos):
    params = decode_particle(pos)
    clf = RandomForestClassifier(
        **params,
        random_state=42,
        n_jobs=-1
    )
    clf.fit(X_sample, y_sample)
    y_pred = clf.predict(X_val)
    return f1_score(y_val, y_pred)

In [ ]:
#Initialise Swarm
positions  = np.random.uniform(BOUNDS_LOW, BOUNDS_HIGH, (N_PARTICLES, N_DIMS))
velocities = np.random.uniform(-1, 1, (N_PARTICLES, N_DIMS))

personal_best_pos   = positions.copy()
personal_best_score = np.array([fitness(p) for p in positions])

global_best_idx   = np.argmax(personal_best_score)
global_best_pos   = personal_best_pos[global_best_idx].copy()
global_best_score = personal_best_score[global_best_idx]

print(f"Initial best F1: {global_best_score:.4f}")
print(f"Initial best params: {decode_particle(global_best_pos)}")

#Run PSO
history = []
start_time = time.time()

print("=" * 50)
print("Running PSO...")
print("=" * 50)

for it in range(N_ITERATIONS):
    it_start = time.time()

    for i in range(N_PARTICLES):
        r1 = np.random.rand(N_DIMS)
        r2 = np.random.rand(N_DIMS)

        # Update velocity
        velocities[i] = (
            W  * velocities[i]
            + C1 * r1 * (personal_best_pos[i] - positions[i])
            + C2 * r2 * (global_best_pos       - positions[i])
        )

        # Update position
        positions[i] = np.clip(
            positions[i] + velocities[i],
            BOUNDS_LOW, BOUNDS_HIGH
        )

        # Evaluate
        score = fitness(positions[i])

        # Update personal best
        if score > personal_best_score[i]:
            personal_best_score[i] = score
            personal_best_pos[i]   = positions[i].copy()

        # Update global best
        if score > global_best_score:
            global_best_score = score
            global_best_pos   = positions[i].copy()

    history.append({
        "iteration": it + 1,
        "best_f1": global_best_score,
        "avg_f1": np.mean(personal_best_score)
    })

    it_time = time.time() - it_start
    print(f"Iter {it+1:2d}/{N_ITERATIONS} | Best F1: {global_best_score:.4f} | "
          f"Avg F1: {np.mean(personal_best_score):.4f} | "
          f"Time: {it_time:.1f}s")

total_time = time.time() - start_time
best_params = decode_particle(global_best_pos)

print(f"\nPSO completed in {total_time:.1f}s")
print(f"Best F1 (val): {global_best_score:.4f}")
print(f"Best params:   {best_params}")


In [ ]:

#Retrain on Full Data with Best Hyperparameters
print("\nRetraining on full training set...")
final_model = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1
)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)



In [ ]:
# Cell 10 — Evaluate
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

print("=" * 45)
print("PSO + RF RESULTS")
print("=" * 45)
print(f"Best Hyperparameters: {best_params}")
print(f"Accuracy:             {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision:            {precision_score(y_test, y_pred):.4f}")
print(f"Recall (Det. Rate):   {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score:             {f1_score(y_test, y_pred):.4f}")
print(f"False Positive Rate:  {fp / (fp + tn):.4f}")
print(f"Features Used:        {X_train.shape[1]} (all — PSO tunes hyperparams only)")
print(f"\nConfusion Matrix:\n{cm}")

# Cell 11 — Convergence Plot
history_df = pd.DataFrame(history)

plt.figure(figsize=(8, 5))
plt.plot(history_df["iteration"], history_df["best_f1"], "b-o", label="Global Best F1")
plt.plot(history_df["iteration"], history_df["avg_f1"], "r--", label="Avg Personal Best F1")
plt.xlabel("Iteration")
plt.ylabel("F1 Score")
plt.title("PSO Convergence Curve")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("pso_convergence.png", dpi=150)
plt.show()

# Cell 12 — Save Results
pso_results = {
    "method": "PSO",
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "fpr": fp / (fp + tn),
    "n_features": X_train.shape[1],
    "runtime_s": total_time,
    **best_params
}

pd.DataFrame([pso_results]).to_csv(base / "data/processed/pso_results.csv", index=False)
print("PSO results saved.")